In [0]:
# === J4 - Silver cleaning test setup ===
import sys
import os

# Path to your repo on Databricks
repo_root = "/Workspace/Users/salmacharki721@gmail.com/cybersecurity-data-pipeline"
sys.path.insert(0, repo_root)

# Reload modules
import importlib
for mod in ['src.ingestion', 'src.cleaning']:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

# Imports
from src.ingestion import (
    generate_sample_dataset,
    ingest_csv_to_bronze_memory,
)
from src.cleaning import clean_bronze_to_silver

# Spark session
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("j4-silver-cleaning").getOrCreate()

print(f"✓ Spark version: {spark.version}")
print(f"✓ src.ingestion and src.cleaning imported successfully")

In [0]:
# === Generate sample and ingest to Bronze (in memory) ===

raw_path = f"{repo_root}/data/raw/"
os.makedirs(raw_path, exist_ok=True)

# Only generate if the sample doesn't exist yet
if not os.path.exists(os.path.join(raw_path, "cic_sample.csv")):
    generate_sample_dataset(raw_path, n_rows=10000)
    print("✓ Sample dataset generated")
else:
    print("✓ Sample dataset already exists")

# Bronze ingestion (in memory)
df_bronze = ingest_csv_to_bronze_memory(spark, raw_path)

print(f"\n✓ Bronze DataFrame ready")
print(f"  Rows: {df_bronze.count()}")
print(f"  Columns: {df_bronze.columns}")

In [0]:
# === Inspect Bronze before cleaning ===

print("--- Bronze schema ---")
df_bronze.printSchema()

print("\n--- Preview (first 5 rows) ---")
df_bronze.show(5, truncate=False)

print("\n--- Label distribution (Bronze) ---")
df_bronze.groupBy("label").count().orderBy(F.col("count").desc()).show()

print("\n--- Null count per column (Bronze) ---")
from pyspark.sql.functions import col, sum as spark_sum, count as spark_count, when

null_counts = df_bronze.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_bronze.columns
])
null_counts.show(truncate=False)

print("\n--- Duplicates count (Bronze) ---")
total = df_bronze.count()
distinct = df_bronze.distinct().count()
print(f"  Total rows: {total}")
print(f"  Distinct rows: {distinct}")
print(f"  Duplicates: {total - distinct}")

In [0]:
# === Run Bronze -> Silver cleaning pipeline ===

print("Starting cleaning pipeline...")
print("=" * 60)

df_silver = clean_bronze_to_silver(df_bronze)

print("=" * 60)
print(f"\n✓ Silver DataFrame ready")
print(f"  Rows: {df_silver.count()}")
print(f"  Columns ({len(df_silver.columns)}): {df_silver.columns}")

In [0]:
# === Inspect Silver after cleaning ===
from pyspark.sql.functions import col, sum as spark_sum, when  # ← AJOUTE CETTE LIGNE

print("--- Silver schema ---")
df_silver.printSchema()

print("\n--- Preview (first 5 rows) ---")
df_silver.show(5, truncate=False)

print("\n--- Label distribution (Silver) ---")
df_silver.groupBy("label").count().orderBy(F.col("count").desc()).show()

print("\n--- Attack category distribution (Silver) ---")
df_silver.groupBy("attack_category").count().orderBy(F.col("count").desc()).show()

print("\n--- is_attack distribution (Silver) ---")
df_silver.groupBy("is_attack").count().orderBy(F.col("is_attack")).show()

print("\n--- Null count per column (Silver) ---")
null_counts_silver = df_silver.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_silver.columns
])
null_counts_silver.show(truncate=False)

In [0]:
# === Check available catalogs ===
print("--- Available catalogs ---")
spark.sql("SHOW CATALOGS").show(truncate=False)

print("\n--- Current catalog ---")
spark.sql("SELECT current_catalog()").show()

print("\n--- Available databases (schemas) ---")
spark.sql("SHOW DATABASES").show(truncate=False)

In [0]:
# === Create schema and volume in the workspace catalog ===

# Switch to workspace catalog
spark.sql("USE CATALOG workspace")
print("✓ Current catalog set to: workspace")

# Create the schema (database)
spark.sql("CREATE SCHEMA IF NOT EXISTS cybersecurity")
print("✓ Schema 'cybersecurity' created (or already exists)")

# Create the volume
spark.sql("CREATE VOLUME IF NOT EXISTS cybersecurity.raw_data")
print("✓ Volume 'cybersecurity.raw_data' created (or already exists)")

# Verify
print("\n--- Volumes in workspace.cybersecurity ---")
spark.sql("SHOW VOLUMES IN workspace.cybersecurity").show(truncate=False)

print("\n--- Volume details ---")
spark.sql("DESCRIBE VOLUME workspace.cybersecurity.raw_data").show(truncate=False)